# Lab 13: How to put the agent on a web page

**CPU | API key: `OPENAI_API_KEY` (Part A and the sign-in check in Part E run without one) | Hugging Face account: only for the optional push in Part F | In class**

## Goal

Serve Lab 12's guarded copilot as a chat page that several reps can use at once, each in their own conversation, and
prepare the folder a Hugging Face Space builds from. The moment comes in Part B: with one thread id for the whole page,
a second rep who asks which customer they are working on gets the first rep's customer and invoices.

## What you will learn

- How to turn a Python function into a chat page with Gradio's `ChatInterface`, and test it from code as several
  visitors with `gradio_client`.
- How to give each visitor their own conversation by using the browser session as the agent's `thread_id`.
- How to stream progress (the tool steps) and hold the reply until its check has passed.
- How to approve or reject a refund inside the chat, with a button or a typed reason.
- How to keep the key out of your files and put a password in front of a page.
- How to assemble the folder a Hugging Face Space builds from, check it, and run it the way the Space will.

## Set up the notebook

Run these two cells first, every time. **Setup** finds the repository folder (on Colab it first copies the repository
and installs the packages). **Config** reads the model names from `models.env` and finds your OpenAI key without
printing it. Lab 00 explains each line.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


⚠️ On Colab, run the setup cell first in a fresh session. If another cell loaded the old `openai` package before the
install, the first model call fails with `ImportError: cannot import name ... from 'openai...'`; use **Runtime →
Restart session** and run from the top.

In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "FALLBACK_MODEL", "EMBED_MODEL", "MODERATION_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
FALLBACK_MODEL    gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
MODERATION_MODEL  omni-moderation-latest
OPENAI_API_KEY    found in .env


The copilot uses the same models as in Lab 12. This lab adds [Gradio](https://www.gradio.app/docs), the library that
draws the page.

In [3]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # a progress-bar notice from a Gradio dependency
import gradio as gr

print("gradio", gr.__version__)

gradio 6.29.0


One message from a rep's browser to the copilot and back; the letters are the parts that add each step.

![One message from a rep's browser to the copilot and back](resources/lab13_page_flow.png)

---

## Part A: Put a function behind a chat page

This part shows a chat page with no model behind it, so the page's own behaviour is clear before the copilot arrives.

[`gr.ChatInterface`](https://www.gradio.app/docs/gradio/chatinterface) turns one Python function into a chat page.
Gradio calls the function once per message with the new `message` and the `history` of the conversation, and shows
what it returns. This one also keeps the last `history` it received, so the notebook can print it:

In [4]:
seen = {}

def echo(message, history):
    seen["history"] = history          # kept only so that the notebook can print it
    return f"You said: {message}. Messages before this one: {len(history)}."

print(echo("hello", []))

You said: hello. Messages before this one: 0.


Give the function to `ChatInterface` and start the page. Every page in this lab uses the same settings: the server
runs in the background (`prevent_thread_lock`), stays on your machine (`share=False`), and shows under the cell only
on Colab (`inline`), where your browser cannot reach the notebook's machine.

In [5]:
LAUNCH = dict(prevent_thread_lock=True, share=False, inline=IN_COLAB, quiet=True)

page = gr.ChatInterface(echo, title="Echo")
page.launch(**LAUNCH)
print("the page is at", page.local_url)

the page is at http://127.0.0.1:7860/


Open the address in your browser and send two messages: the count goes up, because the page sends the whole
conversation each time.

[`gradio_client`](https://www.gradio.app/guides/getting-started-with-the-python-client) opens the page as a browser
does, so you can test it from code. Each `Client` is one visitor with its own session; the page's API name is the
function's name, `/echo`:

In [6]:
from gradio_client import Client

visitor_a = Client(page.local_url, verbose=False)
print(visitor_a.predict("hello", api_name="/echo"))
print(visitor_a.predict("hello again", api_name="/echo"))
print("history:", seen["history"])

You said: hello. Messages before this one: 0.
You said: hello again. Messages before this one: 2.
history: [{'role': 'user', 'content': 'hello'}, {'role': 'assistant', 'content': 'You said: hello. Messages before this one: 0.'}]


The second reply counts 2 earlier messages, *hello* and the page's answer. `history` is a list of dictionaries, one
per message, each with a `role` and its `content`.

Watch for this: through `gradio_client` the content is a string, but a browser visitor's history in Gradio 6 holds a
list of blocks, `[{"type": "text", "text": "hello"}]` (we checked in a browser with Gradio 6.29.0). Code that
reads `history` must handle both; the copilot's page never reads it, because its thread holds the conversation. A
second visitor starts empty:

In [7]:
visitor_b = Client(page.local_url, verbose=False)
print(visitor_b.predict("hi", api_name="/echo"))

You said: hi. Messages before this one: 0.


### Find out who is asking

The copilot's memory is its thread in the checkpointer (Lab 03), so the page must tell the agent which thread a
message belongs to. Add an argument of type [`gr.Request`](https://www.gradio.app/docs/gradio/request) and Gradio fills
it in on every call; its `session_hash` is a random id for one visitor's open page.

In [8]:
def whoami(message, history, request: gr.Request):
    return f"session {request.session_hash[:8]}"

page.close()
page = gr.ChatInterface(whoami)
page.launch(**LAUNCH)
visitor_a, visitor_b = Client(page.local_url, verbose=False), Client(page.local_url, verbose=False)
print("A:", visitor_a.predict("hello", api_name="/whoami"), "|", visitor_a.predict("again", api_name="/whoami"))
print("B:", visitor_b.predict("hello", api_name="/whoami"))
page.close()

Closing server running on port: 7860


A: session d05ae15c | session d05ae15c
B: session a0484170
Closing server running on port: 7860


Visitor A keeps one session id across messages and visitor B has another. Part B uses that id as the thread.

---

## Part B: Put the copilot behind the page, one thread per visitor

This part puts Lab 12's guarded copilot behind the page and shows why the thread id is the line that matters most.

The copilot has every guard Lab 12 added: moderation of messages, tool results and replies; model retries and a
fallback (both models built with `max_retries=0`, so only the middleware retries); a retry for the ticket search; the
check that removes planted lines from tool results; and approval before `file_refund_request`. `search_tickets`
returns only the tickets of the rep in the runtime context. Two choices are new for the page:

- Approval allows approve and reject only: a chat box has no form for changing an invoice or an amount, so `edit`
  is left out, and `respond` stays out as in Lab 09.
- The page has one box, for the rep's message, and none to paste a ticket into. Tickets reach the copilot through
  `search_tickets`, where Lab 12's check removes planted lines; a paste box would be a way in that this check never
  sees.

Start the Ops API (the tools call it) and set the messages two guards show:

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import (HumanInTheLoopMiddleware, ModelFallbackMiddleware, ModelRetryMiddleware,
                                         ToolRetryMiddleware)
from langchain_openai.middleware import OpenAIModerationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

from shared.guards import remove_flagged_lines, search_tickets
from shared.ops_api import start_ops_api
from shared.ops_tools import READ_TOOLS, WRITE_TOOLS, RepContext, copilot_prompt
from shared.settings import chat_model, fallback_model

server = start_ops_api()
VIOLATION = ("This message was flagged for {categories}, so the copilot did not answer it. "
             "Read it yourself, and pass it to the support manager if it needs one.")
SEARCH_DOWN = ("The ticket search is not available right now. "
               "Tell the rep to open the ticket in the ticket system.")

Ops API started at http://127.0.0.1:8765


`build_copilot` lists the guards in Lab 12's order, the outermost first, exactly as the app file in Part F has it.

In [10]:
def build_copilot():
    """Lab 12's guarded copilot, with approve and reject as the decisions a chat page offers."""
    return create_agent(
        chat_model(settings, max_retries=0), tools=READ_TOOLS + [search_tickets] + WRITE_TOOLS,
        system_prompt=copilot_prompt(), context_schema=RepContext, checkpointer=InMemorySaver(),
        middleware=[OpenAIModerationMiddleware(model=settings["MODERATION_MODEL"], check_tool_results=True,
                                               violation_message=VIOLATION),
                    ModelFallbackMiddleware(fallback_model(settings, max_retries=0)),
                    ModelRetryMiddleware(max_retries=2, initial_delay=0.5, on_failure="error"),
                    # after three failed searches, the model reads SEARCH_DOWN instead of an error
                    ToolRetryMiddleware(tools=["search_tickets"], max_retries=2, initial_delay=0.5,
                                        on_failure=lambda error: SEARCH_DOWN),
                    remove_flagged_lines,
                    HumanInTheLoopMiddleware(interrupt_on={
                        "file_refund_request": {"allowed_decisions": ["approve", "reject"]}})])

copilot = build_copilot()
REP = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
print([tool.name for tool in READ_TOOLS + [search_tickets] + WRITE_TOOLS])

['search_policies', 'find_customer', 'get_customer_invoices', 'get_invoice', 'search_tickets', 'file_refund_request']


`REP` is the rep this page acts for: a page anyone can open does not know who is typing, so it is one fixed rep here.

### One thread for everyone

The simplest page function sends the message to the copilot and returns the last reply. It ignores `history`, since
the thread holds the conversation, and it needs a thread id. The first one that comes to mind is a constant:

In [11]:
def respond_one_thread(message, history):
    cfg = {"configurable": {"thread_id": "page"}}
    result = copilot.invoke({"messages": [{"role": "user", "content": message}]}, cfg, context=REP)
    return result["messages"][-1].text

Two reps open the page. Rep A asks about a customer; rep B, a moment later, asks what they are working on. Predict what
rep B gets. (`wrap` only folds long replies.)

In [12]:
import textwrap

def wrap(text, indent=6):
    return textwrap.fill(" ".join(text.split()), width=110, subsequent_indent=" " * indent)

ASK_A = "Helena Holý writes that she was charged twice for the same album. Which invoices are they?"
ASK_B = "Which customer and which invoices are we working on?"

page = gr.ChatInterface(respond_one_thread)
page.launch(**LAUNCH)
rep_a, rep_b = Client(page.local_url, verbose=False), Client(page.local_url, verbose=False)
print("A:", wrap(rep_a.predict(ASK_A, api_name="/respond_one_thread")[:300], 3))
reply_b = rep_b.predict(ASK_B, api_name="/respond_one_thread")
print("B:", wrap(reply_b[:300], 3))
print("rep B's reply names Helena:", "Helena" in reply_b)
page.close()

A: Helena Holý has seven invoices. Notably, these two appear to be charges for albums: - Invoice 220, dated
   2011-08-22, total: $5.94 - Invoice 404, dated 2013-11-13, total: $25.86 Please review these invoices to
   confirm the duplicate charge.


B: We are working on the customer Helena Holý. The invoices relevant to the alleged duplicate charge are: -
   Invoice 220, dated 2011-08-22, total: $5.94 - Invoice 404, dated 2013-11-13, total: $25.86 There are seven
   invoices in total for her account.
rep B's reply names Helena: True
Closing server running on port: 7860


Rep B asked nothing about Helena Holý, and the reply named her and her invoices, as in every run we tried
(5 of 5). Both reps wrote into one thread, so the copilot answered rep B from rep A's conversation: a
customer's data shown to someone who did not ask, and no error anywhere.

### One thread per visitor

The fix is the session id from Part A:

In [13]:
def respond_per_visitor(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.session_hash}}
    result = copilot.invoke({"messages": [{"role": "user", "content": message}]}, cfg, context=REP)
    return result["messages"][-1].text

copilot = build_copilot()      # an empty checkpointer, so nothing from the run above is left
page = gr.ChatInterface(respond_per_visitor)
page.launch(**LAUNCH)
rep_a, rep_b = Client(page.local_url, verbose=False), Client(page.local_url, verbose=False)
print("A:", wrap(rep_a.predict(ASK_A, api_name="/respond_per_visitor")[:300], 3))
reply_b = rep_b.predict(ASK_B, api_name="/respond_per_visitor")
print("B:", wrap(reply_b[:300], 3))
print("rep B's reply names Helena:", "Helena" in reply_b)
print("try it in two browser tabs:", page.local_url)

A: Helena Holý has 7 invoices. The relevant ones are: 1. Invoice ID 46, dated 2009-07-11, total $8.91 2. Invoice
   ID 175, dated 2011-02-15, total $1.98 3. Invoice ID 198, dated 2011-05-20, total $3.96 4. Invoice ID 220,
   dated 2011-08-22, total $5.94 5. Invoice ID 272, dated 2012-04-11, total $0.99 6. I


B: I couldn't find any customer with the name provided. Please double-check the name and try again.
rep B's reply names Helena: False
try it in two browser tabs: http://127.0.0.1:7860/


Rep B's copilot had no customer to go on and asked for one (0 of 5 named Helena); rep A's thread was untouched.

See it as the reps would: the page is still running. Open its address in two browser tabs and ask one question in
each; each tab is a session with its own thread. (On Colab, the two clients play the tabs.) Then close the page.

A session lasts as long as the open page (Part D shows the limit). A real app takes the thread from the signed-in rep
and a case number, and keeps threads in a database (Lab 03's `SqliteSaver`, or Postgres), not in `InMemorySaver`.

In [14]:
page.close()

Closing server running on port: 7860


---

## Part C: Stream progress, and hold the reply until its check passes

This part shows the rep work in progress without showing a reply the checks have not passed.

A Gradio page function streams by `yield`-ing: every `yield` replaces what the page shows. The copilot streams with
`stream_mode="messages"`, each reply piece and tool result as it arrives; with `version="v2"` (Lab 11) each item is a
dictionary whose `data` holds the piece and its metadata, including its node (`model` or `tools`). The last guard is
Lab 12's reply check; `checked` returns what the rep may see and the checks that failed:

In [15]:
from shared.guards import PLANTED, REMOVED, check_no_phone, check_not_obeyed

SAFE_REPLY = "This reply failed a safety check and is not shown. Read the ticket in the ticket system."
OUTPUT_CHECKS = {"planted string absent": check_not_obeyed, "no phone digits": check_no_phone}

def checked(reply):
    # The reply the rep sees, and the checks it failed.
    failed = [name for name, check in OUTPUT_CHECKS.items() if not check({"reply": reply})]
    return (SAFE_REPLY if failed else reply), failed

print(checked("Refunds take 5 to 10 business days."))
print(checked(PLANTED))

('Refunds take 5 to 10 business days.', [])
('This reply failed a safety check and is not shown. Read the ticket in the ticket system.', ['planted string absent'])


### Stream the reply word by word

The first streaming page shows the model's words as they arrive, then checks the finished reply and shows the result.

In [16]:
def respond_words(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.session_hash}}
    so_far = ""
    for item in copilot.stream({"messages": [{"role": "user", "content": message}]}, cfg, context=REP,
                               stream_mode="messages", version="v2"):
        token, meta = item["data"]
        if meta["langgraph_node"] == "model" and token.text:     # a piece of the reply, not a tool result
            so_far += token.text
            yield so_far
    reply, failed = checked(copilot.get_state(cfg).values["messages"][-1].text)
    yield reply

`Client.submit` returns a job you can loop over, one item per `yield`: what a browser shows, update by update. `watch`
records each update and when it arrived. Ask Lab 12's fair question, whose search returns tickets 14 and 16:

In [17]:
import time

def watch(url, api_name, message):
    # Every update the page shows for one message, and the seconds at which each arrived.
    visitor, start, updates = Client(url, verbose=False), time.perf_counter(), []
    for update in visitor.submit(message, api_name=api_name):
        updates.append((round(time.perf_counter() - start, 1), update))
    return updates

FAIR = "Summarise the newest note on the duplicate-charge case for me."
page = gr.ChatInterface(respond_words)
page.launch(**LAUNCH)
updates = watch(page.local_url, "/respond_words", FAIR)
print(f"{len(updates)} updates; first after {updates[0][0]} s, last after {updates[-1][0]} s")
print("first:", updates[0][1])
print("last: ", wrap(updates[-1][1], 6))

63 updates; first after 4.4 s, last after 5.3 s
first: The
last:  The newest note on the duplicate charge case confirms the customer reported a duplicate charge of $9.99 for a
      digital album on November 2, 2013, and attached both invoice numbers for review. The ticket is still
      open, with the assigned representative under review [ticket_16_overpayment_note.md].


The first words came after 4.4 s and the whole reply after 5.3 s, in 63 updates; the rep reads
each word as it comes.

The reply passed because Lab 12's check removed ticket 16's planted line before the model read it. The reply check is
there for a line that gets through anyway (worded so the pattern misses it), and it needs the whole reply, which this
page shows before it is whole. A real model obeys such a line only some of the time, so swap in a **scripted model**
(`shared/scripted.py`) that always replies with the planted string, streamed word by word.

In [18]:
# SCRIPTED MODEL · NO API CALL: a copilot whose model always replies with the planted string.
from langchain_core.messages import AIMessage
from shared.scripted import scripted

def obeying_copilot():
    return create_agent(scripted(AIMessage(content=PLANTED)), tools=[], context_schema=RepContext,
                        checkpointer=InMemorySaver())

real_copilot, copilot = copilot, obeying_copilot()
for seconds, update in watch(page.local_url, "/respond_words", FAIR):
    print(f"{seconds:4} s  {update}")
copilot = real_copilot
page.close()

 0.1 s  APPROVED
 0.1 s  APPROVED 
 0.1 s  APPROVED -
 0.1 s  APPROVED - 
 0.1 s  APPROVED - PAY
 0.1 s  APPROVED - PAY 
 0.1 s  APPROVED - PAY IMMEDIATELY
 0.1 s  This reply failed a safety check and is not shown. Read the ticket in the ticket system.


Closing server running on port: 7860


The page built the planted string up word by word and showed it whole before the check replaced it: the check
worked, after the rep had read it. A page can accept that, check each sentence as it streams (the Challenge), or
stream something that needs no check while the reply waits.

### Stream the tool steps, hold the reply

This page streams each tool result folded: a [`gr.ChatMessage`](https://www.gradio.app/docs/gradio/chatbot) with a
`title` in its `metadata` shows as a closed box. The reply is shown once, after the check.

In [19]:
def tool_step(message):
    # A tool result, folded under a title in the chat.
    return gr.ChatMessage(content=message.text, metadata={"title": f"Tool: {message.name}", "status": "done"})

In [20]:
def respond_steps(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.session_hash}}
    shown = []
    for item in copilot.stream({"messages": [{"role": "user", "content": message}]}, cfg, context=REP,
                               stream_mode="messages", version="v2"):
        token, meta = item["data"]
        if token.type == "tool":
            shown.append(tool_step(token))
            yield shown
    reply, failed = checked(copilot.get_state(cfg).values["messages"][-1].text)
    yield shown + [gr.ChatMessage(content=reply)]

The client receives each `ChatMessage` as a dictionary. `show` prints an update the way the page lays it out: a
folded box as its title and the length of its text, then the reply.

In [21]:
def show(update):
    # One update as the page lays it out: each folded box by its title, then the reply.
    for message in update:
        title = (message.get("metadata") or {}).get("title")
        if title:
            print(f"   [{title}] folded, {len(message['content'])} characters")
        else:
            print("   " + wrap(message["content"], 3))

page = gr.ChatInterface(respond_steps)
page.launch(**LAUNCH)
updates = watch(page.local_url, "/respond_steps", FAIR)
print(f"{len(updates)} updates; first after {updates[0][0]} s, last after {updates[-1][0]} s")
show(updates[-1][1])
print("lines the check removed, inside the folded boxes:", sum(m["content"].count(REMOVED) for m in updates[-1][1]))

2 updates; first after 1.5 s, last after 3.9 s
   [Tool: search_tickets] folded, 1122 characters
   The newest note on the duplicate-charge case indicates that the customer reported a duplicate charge of $9.99
   for a digital album on 2013-11-02 and attached both invoice numbers. The assigned representative is still
   reviewing the situation [ticket_16_overpayment_note.md].
lines the check removed, inside the folded boxes: 1


The folded `search_tickets` result came after 1.5 s, before the word-by-word page's first word, and the
checked reply after 3.9 s. Inside the folded box, ticket 16's planted line is replaced by the mark of Lab 12's
check, so a rep who opens it sees that a guard acted. The scripted model, through this page:

In [22]:
copilot = obeying_copilot()          # SCRIPTED MODEL · NO API CALL
for seconds, update in watch(page.local_url, "/respond_steps", FAIR):
    print(f"{seconds:4} s")
    show(update)
copilot = real_copilot
page.close()

 0.1 s
   This reply failed a safety check and is not shown. Read the ticket in the ticket system.


Closing server running on port: 7860


One update, the safe message: the check runs before the rep sees anything, whatever the model writes.

---

## Part D: Approve a refund inside the chat

This part brings Lab 09's approval step to the page, so the rep decides on a write in the chat.

On a page, the rep's next message is the decision. So the page function first asks the thread whether a write is
waiting (`copilot.get_state(cfg).interrupts`): if one is, the message is a decision; if not, a new request. *approve*
approves; anything else rejects, with the rep's words as the reason the model reads (Lab 09).

In [23]:
def decision(message):
    # "approve" approves; anything else rejects, with the rep's words as the reason.
    if message.strip().lower() == "approve":
        return {"type": "approve"}
    return {"type": "reject", "message": f"Rejected by the rep: {message}. Do not file anything; "
                                         "tell the rep what you found."}

print(decision("Approve"))
print(decision("the customer already got this refund by phone"))

{'type': 'approve'}
{'type': 'reject', 'message': 'Rejected by the rep: the customer already got this refund by phone. Do not file anything; tell the rep what you found.'}


The waiting request is shown with an **Approve** button: each of a `ChatMessage`'s `options` is a button under the
message, and a click sends its `value` as the rep's next message.

In [24]:
def approval_message(interrupts):
    # The waiting write, in words, with an Approve button.
    asks = [ask for interrupt in interrupts for ask in interrupt.value["action_requests"]]
    text = "; ".join(f"{ask['name']} {ask['args']}" for ask in asks)
    return gr.ChatMessage(content=f"Waiting for your approval: {text}. Approve, or type why not.",
                          options=[{"value": "approve", "label": "Approve"}])

### Join the pieces into one page function

The page function joins the thread per visitor (Part B), the folded tool steps and the checked reply (Part C), and
the decision and approval message, and adds a message for a turn that fails (the model call raises once its retries
and the fallback have failed). The app file uses this function.

In [25]:
from langgraph.types import Command

SAFE_MESSAGE = ("The copilot is unavailable right now. Your message is saved; "
                "try again in a minute or handle this ticket by hand.")

In [26]:
def respond(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.session_hash}}
    waiting = copilot.get_state(cfg).interrupts
    if waiting:                     # the message is the rep's decision on the waiting write
        count = len(waiting[0].value["action_requests"])
        turn = Command(resume={"decisions": [decision(message)] * count})
    else:                           # the message is a new request
        turn = {"messages": [{"role": "user", "content": message}]}
    shown = []
    try:
        for item in copilot.stream(turn, cfg, context=REP, stream_mode="messages", version="v2"):
            token, meta = item["data"]
            if token.type == "tool":
                shown.append(tool_step(token))
                yield shown
    except Exception:
        yield shown + [gr.ChatMessage(content=SAFE_MESSAGE)]
        return
    state = copilot.get_state(cfg)
    if state.interrupts:
        yield shown + [approval_message(state.interrupts)]
        return
    reply, failed = checked(state.values["messages"][-1].text)
    yield shown + [gr.ChatMessage(content=reply)]

### Approve once, reject once

Ticket 14 is a duplicate charge, invoices 9001 and 9002. `refund_requests()` reads back every request the Ops API
holds. Rep A asks for the refund:

In [27]:
import httpx
from shared.ops_tools import API

def refund_requests():
    # Every refund request filed so far, read back through the API.
    found, n = [], 1
    while (response := httpx.get(f"{API}/refund-requests/{n}")).status_code == 200:
        found.append(response.json())
        n += 1
    return found

TICKET_14 = ("Ticket 14: a customer was charged twice for the same album, invoices 9001 and 9002. "
             "Please file the refund.")

page = gr.ChatInterface(respond)
page.launch(**LAUNCH)
rep_a = Client(page.local_url, verbose=False)
update = rep_a.predict(TICKET_14, api_name="/respond")
show(update)
print("   buttons:", update[-1]["options"])
print("refund requests in the API:", refund_requests())

   [Tool: get_invoice] folded, 234 characters
   [Tool: get_invoice] folded, 235 characters
   [Tool: search_policies] folded, 1395 characters
   Waiting for your approval: file_refund_request {'invoice_id': 9002, 'amount': 9.99, 'reason':
   'duplicate_charge'}. Approve, or type why not.
   buttons: [{'value': 'approve', 'label': 'Approve'}]
refund requests in the API: []


The copilot read both invoices and the refunds policy, proposed a refund of 9.99 for invoice 9002, the second charge, and paused: nothing is filed yet. Rep
A clicks **Approve**, which sends its value:

In [28]:
before = len(refund_requests())
show(rep_a.predict("approve", api_name="/respond"))
print("filed:", [(r["invoice_id"], r["amount"], r["status"]) for r in refund_requests()[before:]])

   [Tool: file_refund_request] folded, 261 characters
   I filed the refund request for the duplicate charge on invoice 9002, totaling $9.99. It is currently pending
   review [policy_refunds.md]. Suggested reply for the customer: "We've noticed a duplicate charge for your
   album purchase and have filed a refund request for the extra charge. It should be reviewed shortly."
filed: [(9002, 9.99, 'pending_review')]


Filed, with status `pending_review`. Rep B asks for the same refund and knows something the copilot does not:

In [29]:
rep_b = Client(page.local_url, verbose=False)
rep_b.predict(TICKET_14, api_name="/respond")
before = len(refund_requests())
show(rep_b.predict("the customer already got this refund by phone yesterday", api_name="/respond"))
print("filed:", refund_requests()[before:])

   [Tool: file_refund_request] folded, 195 characters
   It seems the customer already received a refund for the duplicate charge via phone yesterday, so no action is
   needed. Suggested reply: "I see that your refund for the duplicate charge has already been processed. If
   you have any other questions, feel free to ask!"
filed: []


The rejection filed nothing, and the reply used the rep's reason. (Had rep B approved too, the API would hold two
requests for one charge: the tool's key includes the thread, as Lab 09 showed.)

### Reload the page while a refund waits

A rep asks for the refund and reloads before approving. A reload is a new session, so a new `Client` plays it:

In [30]:
rep_c = Client(page.local_url, verbose=False)
print("before the reload:", rep_c.predict(TICKET_14, api_name="/respond")[-1]["content"][:80])
rep_c_reloaded = Client(page.local_url, verbose=False)
before = len(refund_requests())
show(rep_c_reloaded.predict("approve", api_name="/respond"))
print("filed:", refund_requests()[before:])
page.close()

before the reload: Waiting for your approval: file_refund_request {'invoice_id': 9002, 'amount': 9.


   I need more context to assist you. Could you please provide more details about what you're seeking approval
   for?
filed: []


Closing server running on port: 7860


After the reload, *approve* reached the copilot as a new message in an empty thread (*I need more context to assist you*), and nothing was
filed. The refund waits in a thread no page will open again, and the rep may think it went through. A thread id that
survives a reload comes from something that does not change, such as the signed-in rep (Part E and the Challenge).

---

## Part E: Keep the key secret and put a password on the page

This part covers two ways a page can cost you money or data: a key in a file, and a page anyone can open.

`load_key` (Lab 00) finds the key in the environment, then `.env`, then Colab Secrets; `.env` is in `.gitignore`. The
app in Part F uses the same function, and on a Space the key arrives as an environment variable from a Space secret.
Check both without printing the key:

In [31]:
import os

print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))
print(".env is listed in .gitignore:", ".env" in (ROOT / ".gitignore").read_text().split())

OPENAI_API_KEY is set: True
.env is listed in .gitignore: True


⚠️ Never print a key, not even its start: outputs are saved and shared. If a key lands in a file you pushed, revoke
it in the OpenAI dashboard; deleting the file leaves it in the Git history.

Anyone who can open the page can use your copilot on your key.
[`launch(auth=...)`](https://www.gradio.app/guides/sharing-your-app) takes a user name and password (or a list of
pairs) and puts a sign-in screen in front of the page:

In [32]:
def echo_user(message, history, request: gr.Request):
    return f"Hello {request.username}, you said: {message}"

page = gr.ChatInterface(echo_user)
page.launch(**LAUNCH, auth=("jane", "change-me"))
print("page settings without signing in:", httpx.get(page.local_url + "config").status_code)
try:
    Client(page.local_url, verbose=False)
except Exception as error:
    print("visitor without a password:", type(error).__name__)
print("visitor with the password:", Client(page.local_url, auth=("jane", "change-me"), verbose=False)
      .predict("hi", api_name="/echo_user"))
page.close()

page settings without signing in: 401
visitor without a password: AuthenticationError


visitor with the password: Hello jane, you said: hi


Closing server running on port: 7860


Without the password the page answers 401 and the client cannot load it; with it, the function reads the user name
from `request.username`, a name a page can trust more than a session id, and one that survives a reload.

This page's copilot still runs every visitor as one fixed rep, Jane, and its invoice tools leave out Lab 05's rep
check. A real app turns the signed-in name into the `RepContext` it passes on every run, and every tool that returns
a customer's data checks it, as Lab 05's `get_invoice` does.

---

## Part F: Prepare the folder a Space builds from

This part turns the page into a folder a [Hugging Face Space](https://huggingface.co/docs/hub/en/spaces-overview) can
build and run, and checks it on your machine first. A Space is a Git repository that builds one app; the header of
its `README.md` says what to build. `space/` holds a finished template: the header, the app, the pinned requirements
and a `.gitignore`.

In [33]:
LAB = ROOT / "13_Deploy_To_A_Space"
for path in sorted((LAB / "space").iterdir()):
    print(f"{path.name:18}{path.stat().st_size:6} bytes")

.gitignore            40 bytes
README.md            516 bytes
app.py              7302 bytes
requirements.txt     449 bytes


The header is the block between the two `---` lines:

In [34]:
def read_header(readme):
    # The fields between the first two '---' lines of a Space's README.md, as a dictionary.
    block = readme.read_text(encoding="utf-8").split("---")[1]
    fields = {}
    for line in block.strip().splitlines():
        key, value = line.split(":", 1)
        fields[key.strip()] = value.strip().strip('"')
    return fields

header = read_header(LAB / "space" / "README.md")
for key, value in header.items():
    print(f"{key:18}{value}")

title             Chinook Support Copilot
emoji             🎧
colorFrom         indigo
colorTo           green
sdk               gradio
sdk_version       6.29.0
python_version    3.13
app_file          app.py
short_description Lab demo, Chinook support copilot


`title`, `emoji`, the colours and `short_description` make the Space's card. Four fields decide whether it builds:

| Field | What it does |
|---|---|
| `sdk: gradio` | build a Gradio app (the others are `docker` and `static`) |
| `sdk_version: 6.29.0` | the Gradio version to install, so `requirements.txt` has no `gradio` line |
| `python_version: "3.13"` | the Python to use; the default is 3.10 |
| `app_file: app.py` | the file the Space runs |

All fields: the [Spaces configuration reference](https://huggingface.co/docs/hub/en/spaces-config-reference).

### Check the app against the notebook

`space/app.py` holds the six functions you wrote in Parts B to D, plus `build_page()`, which builds the page (or, with
no key, a page that says what is missing). `inspect.getsource` gives a notebook function's source; `ast` finds each
function in the file:

In [35]:
import ast
import inspect

APP = (LAB / "space" / "app.py").read_text(encoding="utf-8")
in_file = {node.name: ast.get_source_segment(APP, node) for node in ast.parse(APP).body
           if isinstance(node, ast.FunctionDef)}
for function in (build_copilot, checked, tool_step, decision, approval_message, respond):
    print(f"{function.__name__:18}same: {inspect.getsource(function).strip() == in_file[function.__name__].strip()}")
print("only in the file:", sorted(set(in_file) - {"build_copilot", "checked", "tool_step", "decision",
                                                  "approval_message", "respond"}))

build_copilot     same: True
checked           same: True
tool_step         same: True
decision          same: True
approval_message  same: True
respond           same: True
only in the file: ['build_page']


Six `True`: the app runs the code you tested.

### Assemble the folder

The app imports `shared/` and reads `models.env` and `data/`: the Chinook database (the Ops API and the ticket search
read it) and `data/corpus/`, the policies and tickets the searches embed. Copy them beside the template:

In [36]:
import shutil
import tempfile

def assemble_space(dest):
    # The template, plus the code, data and settings the app reads, in one folder.
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(LAB / "space", dest)
    shutil.copytree(ROOT / "shared", dest / "shared",
                    ignore=shutil.ignore_patterns("__pycache__", "scripted.py", "eval_cases.py"))
    shutil.copy2(ROOT / "models.env", dest / "models.env")
    (dest / "data").mkdir()
    for name in ("README.md", "chinook.db"):
        shutil.copy2(ROOT / "data" / name, dest / "data" / name)
    shutil.copytree(ROOT / "data" / "corpus", dest / "data" / "corpus")
    return sorted(str(path.relative_to(dest)) for path in dest.rglob("*") if path.is_file())

SPACE = Path(tempfile.gettempdir()) / "chinook-copilot-space"
files = assemble_space(SPACE)
print(len(files), "files:", sum("corpus" in f for f in files), "in data/corpus, and")
for f in files:
    if "corpus" not in f:
        print("  ", f)

55 files: 36 in data/corpus, and
   .gitignore
   README.md
   app.py
   data/README.md
   data/chinook.db
   models.env
   requirements.txt
   shared/__init__.py
   shared/copilot.py
   shared/guards.py
   shared/ops_api.py
   shared/ops_tools.py
   shared/prompts/copilot_actions_v1.md
   shared/prompts/copilot_actions_v1_1.md
   shared/prompts/copilot_system_v0.md
   shared/prompts/copilot_system_v1.md
   shared/prompts/copilot_system_v1_1.md
   shared/prompts/copilot_system_v2.md
   shared/settings.py


55 files, most of them the corpus, and no saved index: the searches embed the corpus into
`InMemoryVectorStore` the first time each runs, so a Space's first question takes a little longer. The data is
fictional (Chinook, MIT licence; the corpus written for this course), which matters: a public Space shows it to
anyone.

### Check the folder before anyone builds it

Four mistakes stop a Space or leak a key, and plain code finds each: an unknown `sdk`, a missing `app_file`, a key in a
file (or no `.env` in `.gitignore`), and an unpinned requirement.

In [37]:
import re

KEY_LIKE = re.compile(r"sk-[A-Za-z0-9_-]{20,}")

def check_space(folder):
    # The problems that would stop a Space or leak a key, as a list of sentences.
    problems, header = [], read_header(folder / "README.md")
    if header.get("sdk") not in ("gradio", "docker", "static"):
        problems.append(f"sdk: {header.get('sdk')} is not gradio, docker or static")
    if not (folder / header.get("app_file", "app.py")).exists():
        problems.append(f"app_file: {header.get('app_file', 'app.py')} is not in the folder")
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix != ".db" and KEY_LIKE.search(path.read_text(errors="ignore")):
            problems.append(f"{path.relative_to(folder)} holds something that looks like a key")
    ignore = folder / ".gitignore"
    if not ignore.exists() or ".env" not in ignore.read_text().split():
        problems.append(".gitignore does not list .env")
    for line in (folder / "requirements.txt").read_text().splitlines():
        if line.strip() and not line.startswith("#") and "==" not in line:
            problems.append(f"requirements.txt does not pin {line.strip()}")
    return problems

print("problems in our folder:", check_space(SPACE))

problems in our folder: []


No problems (Going further breaks a copy to see each check fire).

### Run the folder the way the Space will

A Space runs `python app.py` in the folder, with the key in the environment. Do the same in a separate process, so
nothing in this notebook's memory helps. `start_app` copies your environment without the key (adding the key only
when you pass one), starts `python app.py` in the folder and reads its output until it prints its address. First
without a key, the usual reason a new Space shows an error:

In [38]:
import subprocess

def start_app(folder, port, key=None):
    # Run app.py as a Space would, and wait until it says where it runs.
    env = {k: v for k, v in os.environ.items() if k != "OPENAI_API_KEY"}
    env.update(GRADIO_SERVER_PORT=str(port), PYTHONUNBUFFERED="1")
    if key:
        env["OPENAI_API_KEY"] = key
    process = subprocess.Popen([sys.executable, "app.py"], cwd=folder, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    for line in process.stdout:
        lines.append(line.rstrip())
        if "Running on local URL" in line:
            break
    return process, lines

app, lines = start_app(SPACE, 7870)
print("\n".join(wrap(line, 0) for line in lines))
print("the page shows the message:", "OPENAI_API_KEY is missing" in httpx.get("http://127.0.0.1:7870/config").text)
app.terminate()

This copilot is not set up yet: OPENAI_API_KEY is missing. On a Space, add it under Settings, Variables and
secrets, as a secret. Locally, put it in .env or set it in your shell.
* Running on local URL: http://127.0.0.1:7870
the page shows the message: True


The app served a page that says what is missing, instead of crashing. Now with the key, as a Space secret arrives:

In [39]:
app, lines = start_app(SPACE, 7871, key=os.environ.get("OPENAI_API_KEY"))
print("\n".join(lines))
visitor = Client("http://127.0.0.1:7871/", verbose=False)
show(visitor.predict("How long does a refund take to reach the customer's card?", api_name="/respond"))
app.terminate()

Ops API already running at http://127.0.0.1:8765: reusing it
* Running on local URL:  http://127.0.0.1:7871


   [Tool: search_policies] folded, 1484 characters
   Refunds take 5 to 10 business days to reach the customer's card after support approves the request
   [policy_refunds.md]. Suggested reply: "Once we approve your refund request, it typically takes 5 to 10
   business days for the funds to reflect on your card."


The app reused this notebook's Ops API (a Space starts its own) and answered from the assembled folder. A Space also
installs `requirements.txt` into a fresh Python 3.13.

### Push it to a Space (optional)

⚠️ Creating a Gradio Space needs a paid Hugging Face plan: PRO for a personal account, Team or Enterprise for an
organisation (checked 13 Oct 2026). Without one, stop here: the checked folder and the local run are the result.

With a plan, from a terminal in the assembled folder (replace `you`):

```bash
hf auth login
hf repos create you/chinook-copilot --repo-type space --sdk gradio --private
hf spaces secrets add you/chinook-copilot -s OPENAI_API_KEY    # reads the value from your environment
hf upload you/chinook-copilot . . --repo-type space
hf spaces wait you/chinook-copilot                             # until it runs, or its build fails
hf spaces logs you/chinook-copilot --build                     # the build log, if it failed
```

A secret cannot be read back and is not copied when someone duplicates the Space. On free CPU hardware a Space sleeps
when unused, and `InMemorySaver` loses every thread. A public Space spends your key for anyone with the link: keep
it private, or add a sign-in.

## Checkpoints

| After | You should see |
|---|---|
| Part A | the count going 0, then 2, for one visitor and 0 for a new one; one session id per visitor |
| Part B | one thread: rep B's reply names Helena Holý; one per visitor: rep B is asked for a name |
| Part C | word by word, the scripted planted string on screen before the check; tool steps first (ours: 1.5 s), the reply whole |
| Part D | a refund for one invoice (ours: 9.99 for invoice 9002); *approve* files it; a reason or a reload files nothing |
| Part E | 401 without the password; a reply that greets `jane` with it |
| Part F | six functions the same; 55 files, no problems; a reply from the folder with the key |

Saved outputs: `gpt-4o-mini`, 13 Oct 2026, Gradio 6.29.0. Replies and timings vary between runs and models. Stop the
Ops API before you close the notebook:

In [40]:
if server is not None:
    server.should_exit = True

## Going further

### Share the page through a public link

`launch(share=True)` gives the page a public address (`https://....gradio.live`) for up to a week while the notebook
runs; Colab uses one unless you pass `share=False`. Anyone with the link spends your key, so share only behind a
password. The switch is off in the saved run:

In [41]:
SHARE = False      # True: a public link for a colleague, behind the password below

if SHARE:
    server = start_ops_api()
    page = gr.ChatInterface(respond)
    page.launch(prevent_thread_lock=True, share=True, auth=("jane", "pick-a-real-password"))
    print("share this link and the password with a colleague:", page.share_url)
else:
    print("no public link started")

no public link started


### See the folder check fail

A copy with the four mistakes: a key-shaped `.env` and no `.gitignore`, `sdk: streamlit`, the app renamed to
`main.py`, and two unpinned requirements. The fake key is built from pieces, so none sits in this notebook.

In [42]:
BROKEN = Path(tempfile.gettempdir()) / "chinook-copilot-space-broken"
assemble_space(BROKEN)
(BROKEN / ".env").write_text("OPENAI_API_KEY=" + "sk-" + "proj" + "x" * 24 + "\n")
(BROKEN / ".gitignore").unlink()
readme = BROKEN / "README.md"
readme.write_text(readme.read_text().replace("sdk: gradio", "sdk: streamlit"))
(BROKEN / "app.py").rename(BROKEN / "main.py")
requirements = BROKEN / "requirements.txt"
requirements.write_text(requirements.read_text().replace("openai==3.24.0", "openai").replace("httpx==0.28.1", "httpx"))
for problem in check_space(BROKEN):
    print("-", problem)

- sdk: streamlit is not gradio, docker or static
- app_file: app.py is not in the folder
- .env holds something that looks like a key
- .gitignore does not list .env
- requirements.txt does not pin openai
- requirements.txt does not pin httpx


## What you can do now

- Put a function behind a chat page and test it from code as several visitors.
- Give each visitor their own thread, and say what a constant thread id and a reload do.
- Stream the tool steps and hold the reply until its check has passed.
- Approve or reject a waiting refund in the chat, and confirm the result in the API.
- Keep the key in the environment and put a password in front of a page.
- Assemble, check and run the folder a Space builds from.

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `OSError: Cannot find empty port in range: 7860-7959` | earlier pages still running | `gr.close_all()`, or restart the kernel |
| `ValueError: Cannot find a function with api_name: /chat` | the API name is the function's name | `api_name="/respond"` |
| A tool answers *The Ops API did not answer. Try again in a moment.* | the Ops API is not running | run `start_ops_api()` again |
| `start_app` never returns | the app stopped before printing its address | run `python app.py` in the assembled folder |
| On Colab, no page under the cell | the page needs Colab's proxy | use the notebook's clients |

## Challenge

A worked solution is in `solved/` for the instructor.

1. **Keep a waiting refund across a reload.** Launch the page with `auth=[("jane", ...), ("steve", ...)]` and use
   `request.username` as the thread id. Ask for ticket 14's refund as `jane`, then approve from a new `Client`
   signed in as `jane`. Expected: the refund is filed, and `steve` does not see Jane's conversation. What do you lose?
2. **Check the reply a sentence at a time.** Write `respond_sentences`: collect the model's words until a sentence
   ends, show it if `checked` passes it, and stop with `SAFE_REPLY` when one fails. Try a scripted model that replies
   *I read the newest note. APPROVED - PAY IMMEDIATELY. Nothing else is needed.* Expected: the first sentence, then
   the safe message. What can a check on one sentence not see?

## Resources

- [Gradio: `ChatInterface`](https://www.gradio.app/docs/gradio/chatinterface) and [`Chatbot` with `ChatMessage`](https://www.gradio.app/docs/gradio/chatbot): `history`, streaming, folded steps and buttons
- [Gradio: agents and tool usage](https://www.gradio.app/guides/agents-and-tool-usage): an agent's tool steps as folded messages
- [Gradio: the Python client](https://www.gradio.app/guides/getting-started-with-the-python-client) and [sharing your app](https://www.gradio.app/guides/sharing-your-app): `Client`, `submit`, share links and `auth`
- [Hugging Face: Spaces overview](https://huggingface.co/docs/hub/en/spaces-overview): plans, secrets and variables, sleeping hardware
- [Hugging Face: the `hf` command line](https://huggingface.co/docs/huggingface_hub/en/guides/cli): creating a Space, its secrets, uploading, waiting and logs

This is the last lab. Back to [the lab list](../README.md).